# Parte 1. Decretos de emergencia por lluvias

Grupo 9 · 1 de enero–31 de mayo de 2022

Pregunta del trabajo: ¿el Estado declara emergencia donde más llueve?

Este notebook recorre el buscador de gob.pe con Selenium, comprueba la extracción de cada mes, recupera los títulos completos con requests y clasifica las normas. Cada decreto se cuenta una vez dentro de cada departamento mencionado. Las declaratorias y las prórrogas se conservan separadas.

Ejecuta las celdas en orden y utiliza el kernel `pucp_grupo9`. La primera consulta con Selenium corresponde a enero; las siguientes completan febrero–mayo. Los meses completos y verificados se guardan para reanudar el trabajo.

Fuente de las instrucciones: [Assignment 2](https://github.com/alexanderquispe/Diplomado_PUCP/issues/1881).

## 1. Preparación y parámetros

Los archivos finales se guardan en `datos/`. Los controles de calidad y las respuestas de respaldo se guardan en subcarpetas distintas. Los nombres de las columnas siguen la consigna.

In [1]:
from pathlib import Path
from datetime import date, datetime, timezone
from urllib.parse import urlparse, urljoin
from urllib.robotparser import RobotFileParser
import json
import time
import re
import unicodedata
import hashlib
import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display, Markdown
from selenium import webdriver

pd.set_option("display.max_colwidth", None)

GRUPO = 9
ANIO = 2022
FECHA_INICIO = date(ANIO, 1, 1)
FECHA_FIN = date(ANIO, 5, 31)
NAVEGADOR = "chrome"  # Alternativa: "edge".
REUTILIZAR_CACHE = True
PAUSA_SEGUNDOS = 5.0

candidatas = [Path.cwd(), Path.cwd() / "assignment_2"]
RAIZ = next((p for p in candidatas if (p / "scraping_emergencias.ipynb").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abre la carpeta assignment_2 o el repositorio que la contiene.")
DATOS = RAIZ / "datos"
CONTROL = DATOS / "control" / "scraping"
CACHE = DATOS / "cache_scraping"
for carpeta in (DATOS, CONTROL, CACHE):
    carpeta.mkdir(parents=True, exist_ok=True)

DEPARTAMENTOS = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                 "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco", "Ica",
                 "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                 "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                 "San Martín", "Tacna", "Tumbes", "Ucayali"]
assert len(DEPARTAMENTOS) == 25

SESION_HTTP = requests.Session()
SESION_HTTP.headers.update({"User-Agent": "Mozilla/5.0"})
ultima_peticion_http = 0.0

def guardar_json(objeto, destino):
    destino = Path(destino)
    temporal = destino.with_suffix(destino.suffix + ".tmp")
    temporal.write_text(json.dumps(objeto, ensure_ascii=False, indent=2), encoding="utf-8")
    temporal.replace(destino)

def pedir_pagina(url):
    """Consulta pública con pausas y reintentos limitados para fallos temporales."""
    global ultima_peticion_http
    for intento in range(3):
        espera = PAUSA_SEGUNDOS - (time.monotonic() - ultima_peticion_http)
        if espera > 0:
            time.sleep(espera)
        ultima_peticion_http = time.monotonic()
        try:
            respuesta = SESION_HTTP.get(url, timeout=(10, 40))
            if respuesta.status_code in (429, 500, 502, 503, 504) and intento < 2:
                time.sleep(min(30, 5 * (intento + 1)))
                continue
            respuesta.raise_for_status()
            return respuesta
        except (requests.Timeout, requests.ConnectionError):
            if intento == 2:
                raise
            time.sleep(5 * (intento + 1))
    raise RuntimeError("Consulta incompleta: " + url)

display(pd.DataFrame([{"grupo": GRUPO, "inicio": FECHA_INICIO, "fin": FECHA_FIN,
                       "navegador": NAVEGADOR, "pausa_segundos": PAUSA_SEGUNDOS}]))

,grupo,inicio,fin,navegador,pausa_segundos
0,9,2022-01-01,2022-05-31,chrome,5.0


## 2. Descargar, imprimir y revisar robots.txt

En la consulta preparatoria del 4 de octubre de 2026, el grupo general del archivo prohibía `/admin/` y las consultas con el parámetro `sheet`. AhrefsBot tenía una prohibición total. GPTBot indicaba una espera de 5 segundos y OAI-SearchBot una de 2 segundos. La ruta `/busquedas` utilizada en esta tarea quedaba permitida para el grupo general.

La celda siguiente imprime la versión vigente y comprueba la ruta. Las pausas reducen la frecuencia de solicitudes al servidor. Este notebook utiliza 5 segundos entre peticiones a gob.pe, superando los mínimos de 2 segundos entre páginas y 1 segundo entre normas exigidos por la consigna. Si el archivo vigente cambia, revisa esta explicación antes de entregar.

In [2]:
URL_ROBOTS = "https://www.gob.pe/robots.txt"
respuesta_robots = pedir_pagina(URL_ROBOTS)
robots_texto = respuesta_robots.text
if "user-agent:" not in robots_texto.lower():
    raise ValueError("La respuesta recibida no parece un robots.txt válido. Revisa la respuesta HTTP.")
(CONTROL / "robots.txt").write_text(robots_texto, encoding="utf-8")
print(robots_texto)

politica = RobotFileParser()
politica.parse(robots_texto.splitlines())
ruta_consultada = "https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
control_robots = pd.DataFrame([
    {"agente": agente, "permite_busquedas": politica.can_fetch(agente, ruta_consultada),
     "crawl_delay": politica.crawl_delay(agente)}
    for agente in ["*", "GPTBot", "OAI-SearchBot", "AhrefsBot"]
])
display(control_robots)
if not politica.can_fetch("Mozilla/5.0", ruta_consultada):
    raise PermissionError("robots.txt restringe la ruta para este agente; revisa la política antes de continuar.")
guardar_json({"url": URL_ROBOTS, "consulta_utc": datetime.now(timezone.utc).isoformat(),
              "status_http": respuesta_robots.status_code, "pausa_aplicada": PAUSA_SEGUNDOS},
             CONTROL / "consulta_robots.json")

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



,agente,permite_busquedas,crawl_delay
0,*,True,NaN
1,GPTBot,True,5.0
2,OAI-SearchBot,True,2.0
3,AhrefsBot,False,NaN


## 3. Funciones de lectura del buscador

Selenium espera a que el contador y las tarjetas estén disponibles. La extracción recorre los elementos `article` y obtiene el enlace principal, la fecha publicada y el título resumido de cada tarjeta. La paginación sigue controles visibles y detecta páginas repetidas.

Una ausencia de tarjetas se interpreta como un mes vacío únicamente cuando el contador visible indica cero. Los controles se realizan antes de eliminar duplicados o normas ajenas a lluvias.

In [3]:
import calendar
import json
import re
import time
import unicodedata
from datetime import date
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urljoin, urlsplit, urlunsplit

from bs4 import BeautifulSoup
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

In [4]:
MESES_ES = {
    "enero": 1, "ene": 1, "febrero": 2, "feb": 2,
    "marzo": 3, "mar": 3, "abril": 4, "abr": 4,
    "mayo": 5, "may": 5, "junio": 6, "jun": 6,
    "julio": 7, "jul": 7, "agosto": 8, "ago": 8,
    "septiembre": 9, "setiembre": 9, "sep": 9, "sept": 9, "set": 9,
    "octubre": 10, "oct": 10, "noviembre": 11, "nov": 11,
    "diciembre": 12, "dic": 12,
}
PATRON_TOTAL = re.compile(
    r"(?<![\w.,])([0-9]{1,3}(?:[., \u00a0][0-9]{3})+|[0-9]+)\s+resultados?\b",
    flags=re.IGNORECASE,
)


def _espacios(texto):
    return re.sub(r"\s+", " ", str(texto or "")).strip()


def _sin_tildes(texto):
    return "".join(
        c for c in unicodedata.normalize("NFKD", str(texto))
        if not unicodedata.combining(c)
    ).lower()


def parsear_fecha(texto):
    """Convierte una fecha española o numérica válida a YYYY-MM-DD."""
    limpio = _sin_tildes(_espacios(texto))
    limpio = re.sub(r"^publicado\s*:\s*", "", limpio)
    limpio = re.sub(
        r"^(lunes|martes|miercoles|jueves|viernes|sabado|domingo),?\s+", "", limpio
    )
    # La fecha debe comenzar el fragmento; así evitamos leer números del título.
    patron = re.match(
        r"^(\d{1,2})\s+(?:de\s+)?([a-z]+)\.?\s+(?:de\s+)?(\d{4})\b", limpio
    )
    if patron:
        dia, nombre_mes, anio = patron.groups()
        if nombre_mes not in MESES_ES:
            raise ValueError(f"Mes español desconocido en la fecha: {texto!r}")
        return date(int(anio), MESES_ES[nombre_mes], int(dia)).isoformat()
    patron = re.match(r"^(\d{4})[-/](\d{1,2})[-/](\d{1,2})\b", limpio)
    if patron:
        anio, mes, dia = map(int, patron.groups())
        return date(anio, mes, dia).isoformat()
    patron = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{4})\b", limpio)
    if patron:
        dia, mes, anio = map(int, patron.groups())
        return date(anio, mes, dia).isoformat()
    raise ValueError(f"Formato de fecha sin reconocer: {texto!r}")


def canon_url(enlace):
    """Normaliza un enlace público de gob.pe sin cambiar su recurso."""
    partes = urlsplit(urljoin("https://www.gob.pe/", str(enlace).strip()))
    if partes.scheme not in {"https", "http"}:
        raise ValueError(f"Esquema de enlace inesperado: {enlace!r}")
    if partes.hostname not in {"www.gob.pe", "gob.pe"}:
        raise ValueError(f"Dominio ajeno a gob.pe en la tarjeta: {enlace!r}")
    if partes.username or partes.password or partes.port not in {None, 80, 443}:
        raise ValueError(f"Enlace de norma inesperado: {enlace!r}")
    consulta = [
        (clave, valor) for clave, valor in parse_qsl(partes.query, keep_blank_values=True)
        if not clave.lower().startswith("utm_")
        and clave.lower() not in {"fbclid", "gclid"}
    ]
    return urlunsplit((
        partes.scheme.lower(), partes.hostname.lower(),
        partes.path.rstrip("/") or "/", urlencode(consulta, doseq=True), ""
    ))


def extraer_tarjetas(html):
    """Lee todas las tarjetas de normas del HTML renderizado, sin filtrarlas."""
    sopa = BeautifulSoup(html, "html.parser")
    registros = []
    for posicion, articulo in enumerate(sopa.find_all("article"), start=1):
        enlaces = articulo.select("a.link-principal[href*='/normas-legales/']")
        if not enlaces:
            # Respaldo semántico: la ruta de la norma y un texto de enlace presente.
            enlaces = [
                a for a in articulo.find_all("a", href=True)
                if "/normas-legales/" in urlsplit(urljoin("https://www.gob.pe/", a["href"])).path
                and _espacios(a.get_text(" ", strip=True))
            ]
        if not enlaces:
            # Artículos ajenos a los resultados no aportan tarjetas; el conteo final
            # revelará una estructura distinta o resultados de otro contenido.
            continue
        enlaces_por_url = {}
        for enlace in enlaces:
            url = canon_url(enlace["href"])
            enlaces_por_url.setdefault(url, enlace)
        if len(enlaces_por_url) != 1:
            raise ValueError(f"Artículo {posicion}: más de un enlace de norma distinto.")
        url, enlace = next(iter(enlaces_por_url.items()))
        numero = _espacios(enlace.get_text(" ", strip=True))
        if not numero:
            raise ValueError(f"Artículo {posicion}: falta el número de la norma.")
        texto = _espacios(articulo.get_text(" ", strip=True))
        publicado = re.search(r"\bPublicado\s*:\s*", texto, flags=re.IGNORECASE)
        if publicado is None:
            raise ValueError(f"{numero}: falta la etiqueta obligatoria 'Publicado:'.")
        fecha = parsear_fecha(texto[publicado.end():])

        # El enunciado sitúa el resumen en un párrafo después de la publicación.
        descendientes = list(articulo.descendants)
        orden = {id(nodo): indice for indice, nodo in enumerate(descendientes)}
        nodos_publicado = [
            nodo for nodo in descendientes if isinstance(nodo, str)
            and re.search(r"\bPublicado\s*:", str(nodo), flags=re.IGNORECASE)
        ]
        indice_fecha = min((orden[id(n)] for n in nodos_publicado), default=-1)
        candidatos = []
        for parrafo in articulo.find_all("p"):
            contenido = _espacios(parrafo.get_text(" ", strip=True))
            if orden.get(id(parrafo), -1) <= indice_fecha:
                continue
            if not contenido or re.search(r"\bPublicado\s*:", contenido, re.IGNORECASE):
                continue
            if contenido == numero or texto.find(contenido) < publicado.end():
                continue
            # También puede haber un párrafo con solo la fecha y la etiqueta aparte.
            try:
                if parsear_fecha(contenido) == fecha:
                    continue
            except ValueError:
                pass
            candidatos.append(contenido)
        if not candidatos:
            raise ValueError(
                f"{numero}: falta un párrafo de título después de la fecha; revisar el HTML guardado."
            )
        titulo_lista = candidatos[0]
        registros.append({
            "numero": numero, "fecha": fecha,
            "titulo_lista": titulo_lista, "enlace": url,
        })
    return registros

In [5]:
def leer_total_dom(driver):
    """Lee 'N Resultados' del texto visible. Ausencia del rótulo devuelve None."""
    textos = [elemento.text for elemento in driver.find_elements(By.TAG_NAME, "main")
              if elemento.is_displayed()]
    if not textos:
        cuerpos = driver.find_elements(By.TAG_NAME, "body")
        textos = [cuerpos[0].text] if cuerpos else []
    encontrados = {
        int(re.sub(r"[., \u00a0]", "", coincidencia.group(1)))
        for texto in textos for coincidencia in PATRON_TOTAL.finditer(texto)
    }
    if not encontrados:
        # Algunos diseños sitúan el contador fuera de main; aún leemos texto visible.
        cuerpos = driver.find_elements(By.TAG_NAME, "body")
        if cuerpos:
            encontrados = {
                int(re.sub(r"[., \u00a0]", "", coincidencia.group(1)))
                for coincidencia in PATRON_TOTAL.finditer(cuerpos[0].text)
            }
    if len(encontrados) > 1:
        raise ValueError(f"Conteos visibles ambiguos: {sorted(encontrados)}")
    return next(iter(encontrados)) if encontrados else None


def _habilitado(elemento):
    if not elemento.is_displayed() or not elemento.is_enabled():
        return False
    if (elemento.get_attribute("aria-disabled") or "").lower() == "true":
        return False
    if "disabled" in (elemento.get_attribute("class") or "").lower().split():
        return False
    padres_deshabilitados = elemento.find_elements(
        By.XPATH,
        "./ancestor::*[@aria-disabled='true' or "
        "contains(concat(' ', normalize-space(@class), ' '), ' disabled ')]"
    )
    return not padres_deshabilitados


def _enlace_de_busqueda(elemento):
    href = elemento.get_attribute("href")
    if not href:
        return elemento.tag_name.lower() == "button"
    partes = urlsplit(urljoin("https://www.gob.pe/", href))
    return partes.hostname in {"www.gob.pe", "gob.pe"} and partes.path.rstrip("/") == "/busquedas"


def _deduplicar_elementos(elementos):
    return list({elemento.id: elemento for elemento in elementos}.values())


def _control_siguiente(driver, pagina_actual):
    """Devuelve un control observado; nunca construye enlaces de paginación."""
    contenedores = _deduplicar_elementos(driver.find_elements(
        By.CSS_SELECTOR,
        "nav[aria-label*='agin'], nav[aria-label*='ágin'], "
        "[role='navigation'][aria-label*='agin'], [role='navigation'][aria-label*='ágin'], "
        ".pagination, [class*='pagination'], [class*='paginacion']"
    ))
    contenedores = [c for c in contenedores if c.is_displayed()]
    ids_paginacion = {c.id for c in contenedores}
    # rel=next tiene un significado explícito de continuación.
    explicitos = driver.find_elements(By.CSS_SELECTOR, "a[rel~='next'], button[rel~='next']")
    for elemento in driver.find_elements(By.CSS_SELECTOR, "a, button"):
        if not elemento.is_displayed():
            continue
        etiqueta = _espacios(" ".join([
            elemento.text, elemento.get_attribute("aria-label") or "",
            elemento.get_attribute("title") or "",
        ]))
        if re.search(r"\b(siguiente|next)\b", etiqueta, flags=re.IGNORECASE):
            # Excluimos enlaces de contenido y exigimos un destino de búsqueda.
            if elemento.find_elements(By.XPATH, "./ancestor::article"):
                continue
            if elemento.tag_name.lower() == "button":
                ancestros = elemento.find_elements(By.XPATH, "./ancestor::*")
                if elemento.id not in ids_paginacion and not any(
                    ancestro.id in ids_paginacion for ancestro in ancestros
                ):
                    continue
            explicitos.append(elemento)
    explicitos = [e for e in _deduplicar_elementos(explicitos)
                  if _enlace_de_busqueda(e) and _habilitado(e)]
    if explicitos:
        destinos = {e.get_attribute("href") for e in explicitos if e.get_attribute("href")}
        if len(destinos) > 1:
            raise ValueError("Hay varios controles 'Siguiente' con destinos diferentes.")
        return explicitos[0]

    # Un número aislado solo se acepta dentro de una paginación identificada.
    numericos = []
    for contenedor in contenedores:
        actuales = contenedor.find_elements(By.CSS_SELECTOR, "[aria-current='page']")
        for actual in actuales:
            etiqueta_actual = _espacios(actual.text)
            if etiqueta_actual.isdigit() and int(etiqueta_actual) != pagina_actual:
                raise ValueError(
                    f"La página activa es {etiqueta_actual}; se esperaba {pagina_actual}."
                )
        for elemento in contenedor.find_elements(By.CSS_SELECTOR, "a, button"):
            if _espacios(elemento.text) == str(pagina_actual + 1):
                if _enlace_de_busqueda(elemento) and _habilitado(elemento):
                    numericos.append(elemento)
    numericos = _deduplicar_elementos(numericos)
    if numericos:
        destinos = {e.get_attribute("href") for e in numericos if e.get_attribute("href")}
        if len(destinos) > 1:
            raise ValueError("El siguiente número de página tiene destinos diferentes.")
        return numericos[0]
    return None


def _firma(registros):
    return tuple(registro["enlace"] for registro in registros)

In [6]:
def esperar_resultados(driver, timeout=45, firma_anterior=None, pagina_actual=1):
    """Espera contador y tarjetas coherentes; devuelve (total, registros)."""
    estado = {"firma": None, "desde": None, "detalle": "sin lectura del DOM"}

    def listo(navegador):
        try:
            total = leer_total_dom(navegador)
            if total is None:
                estado["detalle"] = "falta el rótulo visible 'N Resultados'"
                estado["firma"] = None
                return False
            registros = extraer_tarjetas(navegador.page_source)
            firma = _firma(registros)
            if total == 0 and registros:
                estado["detalle"] = "contador cero con tarjetas todavía visibles"
                estado["firma"] = None
                return False
            if total > 0 and not registros:
                estado["detalle"] = f"contador {total}, todavía sin tarjetas completas"
                estado["firma"] = None
                return False
            if len(registros) > total:
                estado["detalle"] = "hay más tarjetas en la página que resultados anunciados"
                estado["firma"] = None
                return False
            if firma_anterior is not None and firma == firma_anterior:
                estado["detalle"] = "la firma de la página anterior sigue presente"
                estado["firma"] = None
                return False
            # En primera página, pocos registros sin paginación pueden indicar carga parcial.
            if pagina_actual == 1 and 0 < len(registros) < total:
                if _control_siguiente(navegador, pagina_actual) is None:
                    estado["detalle"] = (
                        f"contador {total}, {len(registros)} tarjetas y sin continuación visible"
                    )
                    estado["firma"] = None
                    return False
            observacion = (total, firma)
            if observacion != estado["firma"]:
                estado["firma"] = observacion
                estado["desde"] = time.monotonic()
                estado["detalle"] = f"contador {total}, {len(registros)} tarjetas; esperando estabilidad"
                return False
            # Dos o más observaciones consistentes evitan aceptar un DOM en transición.
            if time.monotonic() - estado["desde"] < 1.0:
                return False
            return total, registros
        except (StaleElementReferenceException, ValueError) as error:
            estado["detalle"] = str(error)
            estado["firma"] = None
            return False

    try:
        return WebDriverWait(driver, timeout, poll_frequency=0.5).until(listo)
    except TimeoutException as error:
        raise TimeoutException(
            f"La página no llegó a un estado verificable: {estado['detalle']}. "
            "Revise el HTML y la captura del control antes de continuar."
        ) from error

In [7]:
def extraer_mes(driver, year, month, control_dir, pausa=5.0):
    """Recorre enlaces reales de paginación y verifica el total antes de depurar."""
    if pausa < 5:
        raise ValueError("Use una pausa mínima de 5 segundos para este recorrido.")
    inicio = date(int(year), int(month), 1)
    fin = date(inicio.year, inicio.month, calendar.monthrange(inicio.year, inicio.month)[1])
    mes = inicio.strftime("%Y-%m")
    control_dir = Path(control_dir)
    control_dir.mkdir(parents=True, exist_ok=True)
    consulta = urlencode({
        "contenido[]": "normas", "institucion[]": "pcm",
        "term": "estado de emergencia precipitaciones", "sort_by": "recent",
        "desde": inicio.strftime("%d-%m-%Y"), "hasta": fin.strftime("%d-%m-%Y"),
    })
    url_mes = "https://www.gob.pe/busquedas?" + consulta
    pagina = 1
    todos = []
    firmas_vistas = set()
    total_inicial = None
    firma_anterior = None

    def guardar_evidencia(sufijo):
        base = control_dir / f"{mes}_pagina_{pagina:02d}_{sufijo}"
        try:
            base.with_suffix(".html").write_text(driver.page_source, encoding="utf-8")
        except Exception:
            pass
        if sufijo == "error":
            try:
                driver.save_screenshot(str(base.with_suffix(".png")))
            except Exception:
                pass

    try:
        time.sleep(pausa)
        driver.get(url_mes)
        while True:
            total, registros = esperar_resultados(
                driver, firma_anterior=firma_anterior, pagina_actual=pagina
            )
            guardar_evidencia("verificada")
            if total_inicial is None:
                total_inicial = total
            elif total != total_inicial:
                raise ValueError(
                    f"{mes}: el contador cambió de {total_inicial} a {total} entre páginas."
                )
            firma = _firma(registros)
            if firma in firmas_vistas:
                raise ValueError(f"{mes}: la página {pagina} repite una página anterior.")
            firmas_vistas.add(firma)
            for registro in registros:
                publicada = date.fromisoformat(registro["fecha"])
                if not inicio <= publicada <= fin:
                    raise ValueError(
                        f"{mes}: {registro['numero']} tiene fecha {registro['fecha']} fuera del mes."
                    )
            # Se cuenta todo lo que devolvió el buscador: sin deduplicar ni filtrar PCM.
            todos.extend(registros)
            if len(todos) > total_inicial:
                raise ValueError(
                    f"{mes}: se extrajeron {len(todos)} tarjetas y el contador anuncia {total_inicial}."
                )
            siguiente = _control_siguiente(driver, pagina)
            if len(todos) == total_inicial:
                if siguiente is not None:
                    raise ValueError(
                        f"{mes}: se alcanzó el total anunciado pero existe un control siguiente activo."
                    )
                break
            if siguiente is None:
                raise ValueError(
                    f"{mes}: se recuperaron {len(todos)} de {total_inicial}; "
                    "no se encontró una continuación verificable."
                )
            firma_anterior = firma
            time.sleep(pausa)
            siguiente.click()
            pagina += 1
        resultado = {
            "mes": mes, "resultados_totales": total_inicial,
            "resultados_extraidos": len(todos), "paginas": pagina, "registros": todos,
        }
        (control_dir / f"{mes}_resumen.json").write_text(
            json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        return resultado
    except Exception as error:
        guardar_evidencia("error")
        (control_dir / f"{mes}_pagina_{pagina:02d}_error.txt").write_text(
            f"{type(error).__name__}: {error}\nURL solicitada: {url_mes}\n"
            f"Tarjetas acumuladas antes del fallo: {len(todos)}\n",
            encoding="utf-8",
        )
        raise

## 4. Guardar meses completos y recuperar avances

La memoria local se reutiliza únicamente cuando procede de Selenium, conserva todos los resultados del mes y corresponde al año solicitado. Un mes incompleto se consulta de nuevo. Cada ejecución indica si leyó el buscador o recuperó un mes previamente verificado.

In [8]:
def ruta_cache_mes(mes):
    return CACHE / f"mes_{ANIO}_{mes:02d}.json"

def cargar_mes_verificado(mes):
    archivo = ruta_cache_mes(mes)
    if not REUTILIZAR_CACHE or not archivo.is_file():
        return None
    contenido = json.loads(archivo.read_text(encoding="utf-8"))
    if contenido.get("metodo") != "selenium_dom_v1" or contenido.get("anio") != ANIO:
        return None
    resultado = contenido["resultado"]
    registros = resultado["registros"]
    if resultado["resultados_totales"] != resultado["resultados_extraidos"] or len(registros) != resultado["resultados_extraidos"]:
        return None
    if resultado["mes"] != f"{ANIO}-{mes:02d}":
        return None
    return resultado

def crear_navegador():
    if NAVEGADOR == "chrome":
        navegador = webdriver.Chrome()
    elif NAVEGADOR == "edge":
        navegador = webdriver.Edge()
    else:
        raise ValueError('NAVEGADOR debe ser "chrome" o "edge".')
    navegador.set_page_load_timeout(60)
    return navegador

def ejecutar_meses(meses):
    resultados = []
    navegador = None
    try:
        for mes in meses:
            resultado = cargar_mes_verificado(mes)
            if resultado is not None:
                print(f"{ANIO}-{mes:02d}: recuperado de un mes verificado con Selenium.")
            else:
                if navegador is None:
                    navegador = crear_navegador()
                resultado = extraer_mes(navegador, ANIO, mes, CONTROL, pausa=PAUSA_SEGUNDOS)
                if resultado["resultados_totales"] != resultado["resultados_extraidos"]:
                    raise ValueError(f"El mes {mes} quedó incompleto; revisa el control antes de avanzar.")
                guardar_json({"metodo": "selenium_dom_v1", "anio": ANIO,
                              "consulta_utc": datetime.now(timezone.utc).isoformat(),
                              "resultado": resultado}, ruta_cache_mes(mes))
                print(f"{ANIO}-{mes:02d}: extraído y verificado con Selenium.")
            resultados.append(resultado)
    finally:
        if navegador is not None:
            navegador.quit()
    return resultados

## 5. Primera extracción: enero de 2022

La ventana del navegador permite observar la búsqueda. Verifica que la tabla muestre el mismo número de resultados anunciados y extraídos. La URL y los controles HTML quedan registrados en `datos/control/scraping/`.

In [9]:
resultado_enero = ejecutar_meses([1])
display(pd.DataFrame([{k: v for k, v in r.items() if k != "registros"} for r in resultado_enero]))
display(pd.DataFrame(resultado_enero[0]["registros"]).head())

2022-01: extraído y verificado con Selenium.


,mes,resultados_totales,resultados_extraidos,paginas
0,2022-01,8,8,1


,numero,fecha,titulo_lista,enlace
0,Decreto Supremo N.° 011-2022-PCM,2022-01-30,Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la ...,https://www.gob.pe/institucion/pcm/normas-legales/2724291-011-2022-pcm
1,Decreto Supremo N.° 010-2022-PCM,2022-01-29,"Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta e ...",https://www.gob.pe/institucion/pcm/normas-legales/2723437-010-2022-pcm
2,Decreto Supremo N.° 009-2022-PCM,2022-01-28,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca, Loreto y San Martín, por impacto de daños ante la ocurrenci ...",https://www.gob.pe/institucion/pcm/normas-legales/2817913-009-2022-pcm
3,Decreto Supremo N.° 004-2022-PCM,2022-01-25,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honoria de la provincia de Puerto Inca del departamento de H ...",https://www.gob.pe/institucion/pcm/normas-legales/2810643-004-2022-pcm
4,Decreto Supremo N.° 007-2022-PCM,2022-01-18,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,https://www.gob.pe/institucion/pcm/normas-legales/2817884-007-2022-pcm


## 6. Completar febrero–mayo y comprobar los cinco meses

Los conteos de esta tabla corresponden a la extracción del buscador antes de la limpieza. Las reducciones posteriores por duplicados, instituciones o motivo se documentan por separado.

In [10]:
resultados_otros = ejecutar_meses([2, 3, 4, 5])
resultados_meses = resultado_enero + resultados_otros
verificacion_mensual = pd.DataFrame([
    {"mes": r["mes"], "resultados_totales": r["resultados_totales"],
     "resultados_extraidos": r["resultados_extraidos"], "paginas": r["paginas"]}
    for r in resultados_meses
]).sort_values("mes")
display(verificacion_mensual)
verificacion_mensual.to_csv(CONTROL / "verificacion_mensual.csv", index=False, encoding="utf-8-sig")
assert len(verificacion_mensual) == 5
assert (verificacion_mensual["resultados_totales"] == verificacion_mensual["resultados_extraidos"]).all()

registros = [fila for resultado in resultados_meses for fila in resultado["registros"]]
columnas_base = ["numero", "fecha", "titulo_lista", "enlace"]
base = pd.DataFrame(registros, columns=columnas_base)
base.to_csv(CONTROL / "resultados_antes_de_limpieza.csv", index=False, encoding="utf-8-sig")
print("Registros extraídos antes de la limpieza:", len(base))

2022-02: extraído y verificado con Selenium.
2022-03: extraído y verificado con Selenium.
2022-04: extraído y verificado con Selenium.
2022-05: extraído y verificado con Selenium.


,mes,resultados_totales,resultados_extraidos,paginas
0,2022-01,8,8,1
1,2022-02,5,5,1
2,2022-03,5,5,1
3,2022-04,4,4,1
4,2022-05,7,7,1


Registros extraídos antes de la limpieza: 29


## 7. Duplicados e institución responsable

Primero se conserva una fila por enlace canónico. Después se seleccionan los enlaces de la PCM y se comprueba si un mismo número de norma aparece en más de una página. Las fechas contradictorias se detienen para revisión, en lugar de resolverlas mediante una eliminación automática.

In [11]:
def normalizar_texto(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip().casefold()

base["enlace"] = base["enlace"].map(canon_url)
duplicados_enlace = base[base.duplicated("enlace", keep=False)].copy()
duplicados_enlace.to_csv(CONTROL / "duplicados_por_enlace.csv", index=False, encoding="utf-8-sig")
conflictos_enlace = base.groupby("enlace")["fecha"].nunique()
if (conflictos_enlace > 1).any():
    raise ValueError("Un mismo enlace presenta fechas distintas. Revisa duplicados_por_enlace.csv antes de deduplicar.")
n_original = len(base)
unicas = base.drop_duplicates("enlace").copy()
print("Filas repetidas por enlace eliminadas:", n_original - len(unicas))

es_pcm = unicas["enlace"].map(lambda x: "/institucion/pcm/" in urlparse(x).path.lower())
otras_instituciones = unicas.loc[~es_pcm].copy()
otras_instituciones.to_csv(CONTROL / "normas_otras_instituciones.csv", index=False, encoding="utf-8-sig")
print("Normas de otras instituciones retiradas:", len(otras_instituciones))
pcm = unicas.loc[es_pcm].copy()
pcm["clave_norma"] = pcm["numero"].map(normalizar_texto)
duplicados_numero = pcm[pcm.duplicated("clave_norma", keep=False)]
duplicados_numero.to_csv(CONTROL / "duplicados_por_numero.csv", index=False, encoding="utf-8-sig")
conflictos_fecha = pcm.groupby("clave_norma")["fecha"].nunique()
if (conflictos_fecha > 1).any():
    raise ValueError("Un número de norma presenta fechas distintas. Revisa duplicados_por_numero.csv.")
n_antes_numero = len(pcm)
pcm = pcm.drop_duplicates("clave_norma").reset_index(drop=True)
print("Filas repetidas por número eliminadas:", n_antes_numero - len(pcm))
print("Normas PCM que pasarán a lectura completa:", len(pcm))
display(pcm.head())

Filas repetidas por enlace eliminadas: 0
Normas de otras instituciones retiradas: 0
Filas repetidas por número eliminadas: 0
Normas PCM que pasarán a lectura completa: 29


,numero,fecha,titulo_lista,enlace,clave_norma
0,Decreto Supremo N.° 011-2022-PCM,2022-01-30,Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la ...,https://www.gob.pe/institucion/pcm/normas-legales/2724291-011-2022-pcm,decreto supremo n.° 011-2022-pcm
1,Decreto Supremo N.° 010-2022-PCM,2022-01-29,"Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta e ...",https://www.gob.pe/institucion/pcm/normas-legales/2723437-010-2022-pcm,decreto supremo n.° 010-2022-pcm
2,Decreto Supremo N.° 009-2022-PCM,2022-01-28,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca, Loreto y San Martín, por impacto de daños ante la ocurrenci ...",https://www.gob.pe/institucion/pcm/normas-legales/2817913-009-2022-pcm,decreto supremo n.° 009-2022-pcm
3,Decreto Supremo N.° 004-2022-PCM,2022-01-25,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honoria de la provincia de Puerto Inca del departamento de H ...",https://www.gob.pe/institucion/pcm/normas-legales/2810643-004-2022-pcm,decreto supremo n.° 004-2022-pcm
4,Decreto Supremo N.° 007-2022-PCM,2022-01-18,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,https://www.gob.pe/institucion/pcm/normas-legales/2817884-007-2022-pcm,decreto supremo n.° 007-2022-pcm


## 8. Recuperar los títulos completos con requests y BeautifulSoup

El texto resumido del buscador puede cortar la causa de la emergencia. Se utiliza `div.description` de cada norma y se retira únicamente el bloque de archivos descargables. Las descripciones completas se guardan individualmente; una interrupción permite continuar desde las pendientes.

In [12]:
def obtener_titulo_completo(enlace):
    identificador = hashlib.sha256(enlace.encode("utf-8")).hexdigest()[:20]
    destino = CACHE / f"norma_{identificador}.json"
    if REUTILIZAR_CACHE and destino.is_file():
        anterior = json.loads(destino.read_text(encoding="utf-8"))
        if anterior.get("enlace") == enlace and anterior.get("titulo") and anterior.get("selector") == "div.description_v1":
            return anterior["titulo"]
    if not politica.can_fetch("Mozilla/5.0", enlace):
        raise PermissionError("La política vigente restringe esta URL: " + enlace)
    respuesta = pedir_pagina(enlace)
    sopa = BeautifulSoup(respuesta.text, "html.parser")
    descripcion = sopa.find("div", class_="description")
    if descripcion is None:
        (CONTROL / f"error_descripcion_{identificador}.html").write_text(respuesta.text, encoding="utf-8")
        raise ValueError("Falta div.description en " + enlace)
    for bloque in descripcion.select(".institution-document__files, script, style"):
        bloque.decompose()
    titulo = re.sub(r"\s+", " ", descripcion.get_text(" ", strip=True)).strip()
    if not titulo:
        raise ValueError("La descripción está vacía: " + enlace)
    guardar_json({"enlace": enlace, "titulo": titulo, "selector": "div.description_v1",
                  "consulta_utc": datetime.now(timezone.utc).isoformat(), "status_http": respuesta.status_code}, destino)
    return titulo

titulos_completos = []
for i, fila in enumerate(pcm.itertuples(index=False), start=1):
    titulo = obtener_titulo_completo(fila.enlace)
    titulos_completos.append(titulo)
    print(f"[{i}/{len(pcm)}] {fila.numero}")
pcm["titulo"] = titulos_completos
display(pcm[["numero", "titulo_lista", "titulo"]].head())

[1/29] Decreto Supremo N.° 011-2022-PCM
[2/29] Decreto Supremo N.° 010-2022-PCM
[3/29] Decreto Supremo N.° 009-2022-PCM
[4/29] Decreto Supremo N.° 004-2022-PCM
[5/29] Decreto Supremo N.° 007-2022-PCM
[6/29] Decreto Supremo N.° 006-2022-PCM
[7/29] Decreto Supremo N.° 005-2022-PCM
[8/29] Decreto Supremo N.° 002-2022-PCM
[9/29] Decreto Supremo N.° 016-2022-PCM
[10/29] Decreto Supremo N.° 015-2022-PCM
[11/29] Decreto Supremo N.° 014-2022-PCM
[12/29] Decreto Supremo N.° 013-2022-PCM
[13/29] Decreto Supremo N.° 012-2022-PCM
[14/29] Decreto Supremo N.° 029-2022-PCM
[15/29] Decreto Supremo N.° 028-2022-PCM
[16/29] Decreto Supremo N.° 027-2022-PCM
[17/29] Decreto Supremo N.° 030-2022-PCM
[18/29] Resolución de Secretaría de Gestión Pública N.° 003-2022-PCM/SGP
[19/29] Decreto Supremo N.° 041-2022-PCM
[20/29] Decreto Supremo N.° 035-2022-PCM
[21/29] Decreto Supremo N.° 034-2022-PCM
[22/29] Decreto Supremo N.° 032-2022-PCM
[23/29] Decreto Supremo N.° 060-2022-PCM
[24/29] Decreto Supremo N.° 059-20

,numero,titulo_lista,titulo
0,Decreto Supremo N.° 011-2022-PCM,Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la ...,Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la vida de las personas a consecuencia de la COVID-19 y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social
1,Decreto Supremo N.° 010-2022-PCM,"Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta e ...","Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta el 13 de febrero de 2022 ."
2,Decreto Supremo N.° 009-2022-PCM,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca, Loreto y San Martín, por impacto de daños ante la ocurrenci ...","Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca, Loreto y San Martín, por impacto de daños ante la ocurrencia de movimiento sísmico"
3,Decreto Supremo N.° 004-2022-PCM,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honoria de la provincia de Puerto Inca del departamento de H ...","Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honoria de la provincia de Puerto Inca del departamento de Huánuco, en los distritos de Constitución, Palcazú y Puerto Bermúdez de la provincia de Oxapampa del departamento de Pasco y en los distritos de Raimondi, Sepahua y Tahuanía de la provincia de Atalaya del departamento de Ucayali"
4,Decreto Supremo N.° 007-2022-PCM,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto


## 9. Clasificar lluvias, tipo de norma y motivo

La clasificación utiliza el título completo en minúsculas. Se identifica primero una prórroga, porque su descripción puede mencionar también una declaración anterior. Si el título contiene ambos motivos, se registra como `otro` y se señala para revisión; las dos categorías se mantienen mutuamente excluyentes.

Las tablas de revisión muestran los casos `otro`, incluso entre normas que después se excluirán por tratarse de otros eventos. Los registros de revisión permiten explicar la decisión usando el texto real de cada norma.

In [13]:
def clasificar_norma(titulo):
    texto = unicodedata.normalize("NFC", str(titulo)).lower()
    es_lluvia = "lluvia" in texto or "precipitaciones" in texto
    if "prórroga" in texto or "prorroga" in texto:
        tipo = "prorroga"
    elif "declara" in texto:
        tipo = "declara"
    else:
        tipo = "otro"
    peligro = "peligro inminente" in texto
    impacto = "impacto de daños" in texto
    ambos = peligro and impacto
    motivo = "peligro inminente" if peligro and not impacto else "impacto de daños" if impacto and not peligro else "otro"
    return {"es_lluvia": es_lluvia, "tipo": tipo, "motivo": motivo, "ambos_motivos": ambos}

clasificacion = pd.DataFrame([clasificar_norma(t) for t in pcm["titulo"]],
                            columns=["es_lluvia", "tipo", "motivo", "ambos_motivos"])
pcm = pd.concat([pcm.reset_index(drop=True), clasificacion], axis=1)
pcm.to_csv(CONTROL / "normas_pcm_clasificadas.csv", index=False, encoding="utf-8-sig")
revisar = pcm[(pcm["tipo"] == "otro") | (pcm["motivo"] == "otro") | pcm["ambos_motivos"]]
display(revisar[["numero", "es_lluvia", "tipo", "motivo", "ambos_motivos", "titulo"]])
revisar.to_csv(CONTROL / "revision_clasificacion.csv", index=False, encoding="utf-8-sig")
lluvias = pcm.loc[pcm["es_lluvia"].eq(True)].copy()
print("Normas PCM relacionadas con lluvias:", len(lluvias))
display(lluvias[["numero", "fecha", "tipo", "motivo", "titulo"]].head())

revision_lluvias = lluvias[(lluvias["tipo"] == "otro") | (lluvias["motivo"] == "otro") | lluvias["ambos_motivos"]]
display(Markdown(
    f"Se identificaron **{len(revisar)}** registros para revisión entre las normas PCM. "
    f"De ellos, **{len(revision_lluvias)}** corresponden a lluvias. "
    "La categoría `otro` conserva títulos que no contienen una de las expresiones previstas o contienen ambos motivos; "
    "su presencia se informa al interpretar los porcentajes."
))

,numero,es_lluvia,tipo,motivo,ambos_motivos,titulo
0,Decreto Supremo N.° 011-2022-PCM,False,declara,otro,False,Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la vida de las personas a consecuencia de la COVID-19 y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social
1,Decreto Supremo N.° 010-2022-PCM,False,prorroga,otro,False,"Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta el 13 de febrero de 2022 ."
3,Decreto Supremo N.° 004-2022-PCM,False,prorroga,otro,False,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honoria de la provincia de Puerto Inca del departamento de Huánuco, en los distritos de Constitución, Palcazú y Puerto Bermúdez de la provincia de Oxapampa del departamento de Pasco y en los distritos de Raimondi, Sepahua y Tahuanía de la provincia de Atalaya del departamento de Ucayali"
4,Decreto Supremo N.° 007-2022-PCM,False,prorroga,otro,False,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto
5,Decreto Supremo N.° 006-2022-PCM,False,prorroga,otro,False,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata, en los distritos de Fitzcarrald, Manu, Madre de Dios y Huepetuhe de la provincia de Manu y en los distritos de Iñapari, Iberia y Tahuamanu de la provincia de Tahuamanu del departamento de Madre de Dios"
6,Decreto Supremo N.° 005-2022-PCM,False,declara,otro,False,Decreto Supremo que modifica el Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la vida de las personas a consecuencia de la COVID-19 y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta el 30 de enero de 2022 .
7,Decreto Supremo N.° 002-2022-PCM,False,declara,otro,False,"Decreto Supremo que modifica el numeral 8.1 del artículo 8 del Decreto Supremo N° 184-2020-PCM, Decreto Supremo quedeclara Estado de Emergencia Nacional por las graves circunstancias que afectanla vida de las personas a consecuenciade la COVID-19 y establece las medidasque debe seguir la ciudadanía en la nueva convivencia social."
8,Decreto Supremo N.° 016-2022-PCM,False,declara,otro,False,Decreto Supremo que declara Estado de Emergencia Nacional por las circunstancias que afectan la vida y salud de las personas como consecuencia de la COVID-19 y establece nuevas medidas para el restablecimiento de la convivencia social
9,Decreto Supremo N.° 015-2022-PCM,False,declara,otro,False,"Decreto Supremo que modifica el Decreto Supremo N° 184-2020-PCM, Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la vida de las personas a consecuencia de la COVID-19 y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social desde el 14 de febrero de 2022."
11,Decreto Supremo N.° 013-2022-PCM,False,prorroga,otro,False,"Prórroga del Estado de Emergencia en distritos de las provincias de Huanta y La Mar del departamento de Ayacucho; de las provincias de Tayacaja y Churcampa del departamento de Huancavelica; de la provincia de La Convención del departamento de Cusco; de las provincias de Satipo, Concepción y Huancayo del departamento de Junín; y, en el Centro Poblado de Yuveni en el distrito de Vilcabamba de la provincia de La Convención del departamento de Cusco"


Normas PCM relacionadas con lluvias: 1


,numero,fecha,tipo,motivo,titulo
21,Decreto Supremo N.° 032-2022-PCM,2022-04-01,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Ayacucho y Piura, por impacto de daños a consecuencia de intensas precipitaciones pluviales"


Se identificaron **22** registros para revisión entre las normas PCM. De ellos, **0** corresponden a lluvias. La categoría `otro` conserva títulos que no contienen una de las expresiones previstas o contienen ambos motivos; su presencia se informa al interpretar los porcentajes.

### Explicación de los casos revisados

La revisión identificó 22 normas clasificadas con motivo “otro”. Sus descripciones carecen de las expresiones “peligro inminente” e “impacto de daños”, por lo que esta categoría corresponde a la regla textual establecida en el ejercicio. Los 22 registros recibieron es_lluvia = False. Algunos títulos mencionan expresamente la COVID-19; otros describen ámbitos territoriales o medidas específicas y ofrecen información limitada sobre la causa de la emergencia.

La Resolución de Secretaría de Gestión Pública N.° 003-2022-PCM/SGP también recibió tipo = “otro”, porque su descripción corresponde a la adecuación de horarios de atención de los Centros MAC. En algunas normas que modifican disposiciones anteriores, como el Decreto Supremo N.° 011-2022-PCM, la etiqueta “declara” procede de una referencia a otro decreto dentro de la descripción. Estas etiquetas reproducen las reglas de búsqueda de palabras indicadas en la consigna; las normas mencionadas quedan excluidas del subconjunto de lluvias.

El filtro conservó el Decreto Supremo N.° 032-2022-PCM, publicado el 1 de abril de 2022. Su título menciona una declaración de emergencia por impacto de daños a consecuencia de intensas precipitaciones pluviales. Por ello, corresponde clasificarlo como es_lluvia = True, tipo = “declara” y motivo = “impacto de daños”. El subconjunto final presenta cero registros con tipo o motivo “otro”.

## 10. Identificar departamentos sin duplicarlos dentro del decreto

Se busca cada nombre por palabra completa en el título original. Una segunda comparación controla mayúsculas y tildes conservando los límites de palabra. Esta regla evita asignar Ica cuando el texto contiene únicamente Huancavelica. Cada departamento aparece una sola vez en la lista del decreto, aunque su nombre se repita como provincia y como departamento.

Las correcciones manuales se reservan para un título realmente ambiguo y deben identificar la norma, los departamentos y una justificación sustentada en la fuente.

In [14]:
# Dejar vacíos cuando todos los departamentos se identifican correctamente.
CORRECCIONES_DEPARTAMENTOS = {}
JUSTIFICACIONES_DEPARTAMENTOS = {}

def departamentos_del_titulo(titulo):
    original = unicodedata.normalize("NFC", str(titulo))
    normalizado = normalizar_texto(original)
    encontrados = []
    for departamento in DEPARTAMENTOS:
        literal = re.search(r"\b" + re.escape(departamento) + r"\b", original)
        variante = re.search(r"\b" + re.escape(normalizar_texto(departamento)) + r"\b", normalizado)
        if literal or variante:
            encontrados.append(departamento)
    if "Cusco" not in encontrados and re.search(r"\bcuzco\b", normalizado):
        encontrados.append("Cusco")
    return encontrados

listas_departamentos = []
for fila in lluvias.itertuples(index=False):
    departamentos = departamentos_del_titulo(fila.titulo)
    if fila.numero in CORRECCIONES_DEPARTAMENTOS:
        departamentos = CORRECCIONES_DEPARTAMENTOS[fila.numero]
        justificacion = JUSTIFICACIONES_DEPARTAMENTOS.get(fila.numero, "").strip()
        if not justificacion:
            raise ValueError("Falta justificar la corrección de " + fila.numero)
        if not set(departamentos).issubset(DEPARTAMENTOS):
            raise ValueError("La corrección contiene un nombre departamental ajeno al maestro.")
        departamentos = [d for d in DEPARTAMENTOS if d in departamentos]
    listas_departamentos.append(departamentos)
lluvias["departamentos"] = listas_departamentos
auditoria_departamentos = lluvias[["numero", "fecha", "titulo", "enlace", "departamentos"]].copy()
auditoria_departamentos["departamentos"] = auditoria_departamentos["departamentos"].map(lambda x: " | ".join(x))
auditoria_departamentos.to_csv(CONTROL / "departamentos_por_norma.csv", index=False, encoding="utf-8-sig")
display(auditoria_departamentos)

sin_departamento = lluvias[lluvias["departamentos"].map(len).eq(0)]
if len(sin_departamento):
    display(sin_departamento[["numero", "titulo", "enlace"]])
    raise ValueError("Hay normas de lluvias sin departamento identificado. Revisa su fuente y documenta la corrección antes de agregar.")

reales_ica = lluvias[lluvias["titulo"].map(lambda x: "ica" in normalizar_texto(x))]
if len(reales_ica):
    ejemplo = reales_ica.iloc[0]
    display(Markdown(f"**Ejemplo real de la temporada:** {ejemplo['numero']}. "
                     f"Departamentos identificados: {', '.join(ejemplo['departamentos'])}. "
                     "La asignación se basa en palabras completas y en una lista de departamentos únicos por norma."))
elif len(lluvias):
    ejemplo = lluvias.iloc[0]
    display(Markdown(f"**Ejemplo real de la temporada:** {ejemplo['numero']}. "
                     f"Se asigna una vez a cada uno de estos departamentos: {', '.join(ejemplo['departamentos'])}. "
                     "La búsqueda por palabras completas evita coincidencias parciales y el recorrido del maestro evita repeticiones."))
else:
    display(Markdown("La búsqueda verificada devolvió cero normas de lluvias; no existe un ejemplo observado que pueda ilustrarse con este subconjunto."))

,numero,fecha,titulo,enlace,departamentos
21,Decreto Supremo N.° 032-2022-PCM,2022-04-01,"Decreto Supremo que declara el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Ayacucho y Piura, por impacto de daños a consecuencia de intensas precipitaciones pluviales",https://www.gob.pe/institucion/pcm/normas-legales/2929443-032-2022-pcm,Amazonas | Ayacucho | Piura


**Ejemplo real de la temporada:** Decreto Supremo N.° 032-2022-PCM. Se asigna una vez a cada uno de estos departamentos: Amazonas, Ayacucho, Piura. La búsqueda por palabras completas evita coincidencias parciales y el recorrido del maestro evita repeticiones.

### Ejemplo y decisión de identificación

El Decreto Supremo N.° 032-2022-PCM menciona expresamente los departamentos de Amazonas, Ayacucho y Piura. La identificación recorrió el listado de departamentos y buscó cada nombre por palabra completa, mediante límites de palabra en expresiones regulares. El código también controló variantes de mayúsculas y tildes.

Cada departamento se incorporó una sola vez a la lista de la norma. Por tanto, este decreto aporta una declaratoria a Amazonas, una a Ayacucho y una a Piura, mientras que la base de decretos conserva una única fila. La búsqueda por palabra completa distingue “Ica” de su aparición dentro de “Huancavelica”, y la incorporación única de cada departamento evita duplicar su conteo cuando el nombre aparece varias veces en un título.

## 11. Agregar y guardar los dos archivos solicitados

El archivo de decretos conserva una fila por norma. El archivo departamental cuenta cada norma una vez dentro de cada departamento que menciona. Por ello, la suma departamental puede ser mayor que el número de decretos únicos.

In [15]:
columnas_finales = ["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]
decretos_lluvias = lluvias[columnas_finales].copy().sort_values(["fecha", "numero"])
asignaciones = lluvias[["numero", "enlace", "tipo", "departamentos"]].explode("departamentos")
asignaciones = asignaciones.dropna(subset=["departamentos"]).rename(columns={"departamentos": "departamento"})
asignaciones = asignaciones.drop_duplicates(["enlace", "departamento"])
conteos = asignaciones.groupby(["departamento", "tipo"]).size().unstack(fill_value=0)
por_departamento = pd.DataFrame({"departamento": DEPARTAMENTOS}).set_index("departamento")
for tipo, columna in [("declara", "declaratorias"), ("prorroga", "prorrogas")]:
    serie = conteos[tipo] if tipo in conteos.columns else pd.Series(dtype="int64")
    por_departamento[columna] = serie.reindex(DEPARTAMENTOS).fillna(0).astype(int)
por_departamento = por_departamento.reset_index()

assert len(por_departamento) == 25
assert not decretos_lluvias["enlace"].duplicated().any()
decretos_lluvias.to_csv(DATOS / "decretos_lluvias.csv", index=False, encoding="utf-8-sig")
por_departamento.to_csv(DATOS / "decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
asignaciones.to_csv(CONTROL / "asignaciones_norma_departamento.csv", index=False, encoding="utf-8-sig")
guardar_json({"grupo": GRUPO, "anio": ANIO, "inicio": FECHA_INICIO.isoformat(), "fin": FECHA_FIN.isoformat(),
              "generado_utc": datetime.now(timezone.utc).isoformat(), "metodo_busqueda": "Selenium sobre DOM visible",
              "metodo_titulos": "requests y BeautifulSoup; div.description", "registros_busqueda": int(len(base)),
              "normas_pcm_unicas": int(len(pcm)), "normas_lluvias_unicas": int(len(decretos_lluvias)),
              "declaratorias_unicas": int(decretos_lluvias["tipo"].eq("declara").sum()),
              "prorrogas_unicas": int(decretos_lluvias["tipo"].eq("prorroga").sum()),
              "correcciones_departamentos": CORRECCIONES_DEPARTAMENTOS,
              "justificaciones_departamentos": JUSTIFICACIONES_DEPARTAMENTOS}, CONTROL / "resumen_extraccion.json")
display(por_departamento.sort_values(["declaratorias", "prorrogas"], ascending=False))
print("Archivos guardados: decretos_lluvias.csv y decretos_por_departamento.csv.")
print("Guarda este notebook con Ctrl+S para conservar todas las salidas.")

,departamento,declaratorias,prorrogas
0,Amazonas,1,0
4,Ayacucho,1,0
19,Piura,1,0
1,Áncash,0,0
2,Apurímac,0,0
3,Arequipa,0,0
5,Cajamarca,0,0
6,Callao,0,0
7,Cusco,0,0
8,Huancavelica,0,0


Archivos guardados: decretos_lluvias.csv y decretos_por_departamento.csv.
Guarda este notebook con Ctrl+S para conservar todas las salidas.
